In [249]:
import pandas as pd
import re
from collections import Counter

In [250]:
df = pd.read_csv(r"c:\Users\Beyza\Desktop\skillsync\data\fake_job_postings.csv")

In [251]:
df = df.fillna("")

In [252]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z ]", "", str(text).lower())
    return text

In [253]:
df["clean_description"] = df["description"].apply(clean_text)
df["clean_requirements"] = df["requirements"].apply(clean_text)

In [254]:
from collections import Counter
ignored_words = {
    "working", "looking", "following", "requirements", "experience", "skills", 
    "excellent", "strong", "within", "perform", "office", "process", "solutions", 
    "company", "position", "online", "customer", "applications", "verbal", 
    "written", "knowledge", "ability", "including", "candidate", "developing",
    "project", "minimum", "provide", "service", "clients", "school", "degree",
    "relevant", "industry", "marketing", "english", "digital", "understanding",
    "analytical", "quality", "writing", "diploma", "products", "technical"
}
def build_dynamic_skills(df, top_n=200):
    all_text = " ".join(df["clean_requirements"])
    words = all_text.split()
    common_words = Counter(words).most_common(top_n)
    #skill_pool = [word for word, count in common_words if len(word) > 5]
    skill_pool = []
    for word, count in common_words:
        if len(word) > 5 and word not in ignored_words:
            skill_pool.append(word)
    return list(set(skill_pool))
    

In [255]:
extracted_skills = build_dynamic_skills(df)
print(f"Veri setinden {len(extracted_skills)} adet dinamik beceri çekildi.")

Veri setinden 62 adet dinamik beceri çekildi.


In [256]:
print(extracted_skills)

['business', 'system', 'background', 'information', 'education', 'manage', 'analysis', 'support', 'management', 'attention', 'science', 'building', 'through', 'environment', 'similar', 'preferred', 'communication', 'projects', 'technology', 'passion', 'equivalent', 'social', 'client', 'microsoft', 'product', 'creative', 'technologies', 'customers', 'multiple', 'interpersonal', 'people', 'mobile', 'performance', 'university', 'organizational', 'detail', 'testing', 'software', 'communicate', 'should', 'windows', 'effectively', 'application', 'design', 'javascript', 'required', 'computer', 'development', 'related', 'services', 'problem', 'systems', 'highly', 'record', 'managing', 'language', 'complex', 'proven', 'attitude', 'solving', 'professional', 'engineering']


In [257]:
import random
def generate_sample_cvs(df, num_samples=20):
    samples = df[df["requirements"] != ""].sample(num_samples)
    cv_outputs = []

    names = ["Ali", "Ayşe", "Mehmet", "Selin", "Can", "Zeynep", "Burak", "Merve"]
    surnames = ["Yılmaz", "Kaya", "Demir", "Çelik", "Öztürk", "Arslan", "Şahin"]
    
    for i, (idx, row) in enumerate(samples.iterrows()):
        name = f"{random.choice(names)} {random.choice(surnames)}"
        email = f"{name.lower().replace(' ', '.')}@email.com"

        cv_text = f"""
==================================================
           ADAY ÖZGEÇMİŞİ (CV #{i+1})
==================================================
İSİM SOYİSİM: {name}
E-POSTA: {email}
LOKASYON: {row['location'] if row['location'] else 'Belirtilmemiş'}
HEDEF POZİSYON: {row['title']}

PROFESYONEL ÖZET:
{row['description'][:400].strip()}...

DENEYİM & YETKİNLİKLER:
{row['requirements'].strip()}

ŞİRKET KÜLTÜRÜNE UYUM (REFERANS):
{row['company_profile'][:200].strip() if row['company_profile'] else 'N/A'}...
==================================================
"""
        cv_outputs.append(cv_text)
    return cv_outputs

In [258]:
sample_cvs_for_test = generate_sample_cvs(df)

In [259]:
print(sample_cvs_for_test[0])


           ADAY ÖZGEÇMİŞİ (CV #1)
İSİM SOYİSİM: Ali Yılmaz
E-POSTA: ali.yılmaz@email.com
LOKASYON: GB, , London
HEDEF POZİSYON: Personal Assistant

PROFESYONEL ÖZET:
Citymapper is looking for a Personal Assistant to look after the administrative needs of our CEO and team. We are an early-stage startup, expanding quickly, so a successful candidate will have to be a fast, flexible, organized and motivated individual with a can-do attitude to allow to the CEO and team to focus on what matters the most. Part of the remit will also be to ensure the office runs smoo...

DENEYİM & YETKİNLİKLER:
#NAME?

ŞİRKET KÜLTÜRÜNE UYUM (REFERANS):
We believe cities are complicated. And your mobile device should save you from the everyday challenges of living in them.We're a small dedicated team based somewhere in London with backgrounds in tran...



In [260]:
def extract_skills_from_text(text, skill_db):
    text = clean_text(text)
    found = [skill for skill in skill_db if skill in text.split()]
    return found

In [261]:
target_job_requirement = "we need someone with experience in management and communication and technical skills"
target_job_skills = extract_skills_from_text(target_job_requirement, extracted_skills)

print("\n" + "="*60)
print("\n--- TEST AŞAMASI ---")
print(f"HEDEF İŞ İLANINDAKİ BECERİLER: {target_job_skills}")
print("="*60)

for i, cv_content in enumerate(sample_cvs_for_test):
    cv_skills = extract_skills_from_text(cv_content, extracted_skills)
    missing_skills = list(set(target_job_skills) - set(cv_skills))
    try:
        candidate_name = cv_content.split("İSİM SOYİSİM: ")[1].split("\n")[0]
    except:
        candidate_name = f"Aday #{i+1}"

    print(f"ANALİZ EDİLEN: {candidate_name}")
    print(f"Bulunan Beceriler: {cv_skills}")
    print(f"Eksik Beceriler  : {missing_skills}")
    print("-" * 40)




--- TEST AŞAMASI ---
HEDEF İŞ İLANINDAKİ BECERİLER: ['management', 'communication']
ANALİZ EDİLEN: Ali Yılmaz
Bulunan Beceriler: ['mobile', 'should', 'attitude']
Eksik Beceriler  : ['management', 'communication']
----------------------------------------
ANALİZ EDİLEN: Zeynep Çelik
Bulunan Beceriler: ['information', 'communication', 'projects', 'microsoft', 'technologies', 'software', 'javascript', 'development', 'services', 'professional']
Eksik Beceriler  : ['management']
----------------------------------------
ANALİZ EDİLEN: Can Arslan
Bulunan Beceriler: []
Eksik Beceriler  : ['management', 'communication']
----------------------------------------
ANALİZ EDİLEN: Selin Yılmaz
Bulunan Beceriler: ['analysis', 'projects', 'social', 'creative', 'people', 'services', 'highly', 'managing', 'professional']
Eksik Beceriler  : ['management', 'communication']
----------------------------------------
ANALİZ EDİLEN: Can Şahin
Bulunan Beceriler: ['environment', 'communication', 'multiple', 'org

In [262]:
# --- TÜM ANALİZ RAPORUNU DOSYAYA KAYDETME (GÜNCEL) ---

with open("nlp_proje_analiz_raporu.txt", "w", encoding="utf-8") as f:
    f.write("==================================================\n")
    f.write("            NLP PROJE TEST ANALİZ RAPORU\n")
    f.write("==================================================\n")
    f.write(f"ARANAN HEDEF BECERİLER: {target_job_skills}\n")
    f.write("==================================================\n\n")
    
    for i, cv_content in enumerate(sample_cvs_for_test):
        # Becerileri ve eksikleri hesapla
        cv_skills = extract_skills_from_text(cv_content, extracted_skills)
        missing_skills = list(set(target_job_skills) - set(cv_skills))
        
        # CV içeriğinden ismi ayıkla
        try:
            candidate_name = cv_content.split("İSİM SOYİSİM: ")[1].split("\n")[0]
        except:
            candidate_name = f"Aday #{i+1}"
        
        # Dosyaya yazma işlemi
        f.write(f"ANALİZ EDİLEN: {candidate_name} (CV #{i+1})\n")
        f.write(f"Bulunan Beceriler: {cv_skills}\n")
        f.write(f"Eksik Beceriler  : {missing_skills}\n")
        f.write("-" * 40 + "\n")

print("Tebrikler! İsim bazlı detaylı rapor 'nlp_proje_analiz_raporu.txt' dosyasına kaydedildi.")

Tebrikler! İsim bazlı detaylı rapor 'nlp_proje_analiz_raporu.txt' dosyasına kaydedildi.


In [263]:
# --- TÜM ÖRNEK CV'LERİ AYRI BİR DOSYAYA KAYDETME ---

file_name = "aday_cv_havuzu.txt"

with open(file_name, "w", encoding="utf-8") as f:
    f.write("==================================================\n")
    f.write("           SİSTEMDE KAYITLI ADAY CV HAVUZU\n")
    f.write(f"           Toplam Kayıt Sayısı: {len(sample_cvs_for_test)}\n")
    f.write("==================================================\n\n")
    
    for cv_content in sample_cvs_for_test:
        # Mevcut CV içeriğini dosyaya yazdırıyoruz
        f.write(cv_content.strip())
        f.write("\n\n" + "*"*50 + "\n\n") # CV'ler arasına belirgin bir ayraç koyar

print(f"Başarılı! {len(sample_cvs_for_test)} adet detaylı CV '{file_name}' dosyasına kaydedildi.")

Başarılı! 20 adet detaylı CV 'aday_cv_havuzu.txt' dosyasına kaydedildi.
